In [30]:
!pip install -U datasets --quiet

In [31]:
from huggingface_hub import hf_hub_download
from datasets import Dataset, DatasetDict

repo = "tner/conll2003"

def load_split(filename):
    path = hf_hub_download(repo, filename, repo_type="dataset")
    return Dataset.from_json(path)

train = load_split("dataset/train.json")
val = load_split("dataset/valid.json")
test = load_split("dataset/test.json")

dataset = DatasetDict({'train': train, 'validation': val, 'test': test})
print(dataset)



DatasetDict({
    train: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 3453
    })
})


In [32]:
sample = dataset['train'][0]
print(sample["tokens"])
print(sample["tags"])

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
[1, 0, 2, 0, 0, 0, 2, 0, 0]


In [33]:
# 加载 label.json
import json
from huggingface_hub import hf_hub_download

path = hf_hub_download("tner/conll2003", "dataset/label.json", repo_type="dataset")
with open(path) as f:
    label_map = json.load(f)
print(label_map)

{'O': 0, 'B-ORG': 1, 'B-MISC': 2, 'B-PER': 3, 'I-PER': 4, 'B-LOC': 5, 'I-ORG': 6, 'I-MISC': 7, 'I-LOC': 8}


In [34]:
id2label = {v: k for k, v in label_map.items()}
label2id = label_map
print(id2label)
print(label2id)

{0: 'O', 1: 'B-ORG', 2: 'B-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-LOC', 6: 'I-ORG', 7: 'I-MISC', 8: 'I-LOC'}
{'O': 0, 'B-ORG': 1, 'B-MISC': 2, 'B-PER': 3, 'I-PER': 4, 'B-LOC': 5, 'I-ORG': 6, 'I-MISC': 7, 'I-LOC': 8}


In [35]:
label_names = [id2label[i] for i in range(len(id2label))]
print(label_names)

['O', 'B-ORG', 'B-MISC', 'B-PER', 'I-PER', 'B-LOC', 'I-ORG', 'I-MISC', 'I-LOC']


In [36]:
from transformers import AutoTokenizer

model_name = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokens = tokenizer(
    sample['tokens'],
    is_split_into_words=True,
    truncation=True,
    max_length=128,
)
print(tokens)

{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [37]:
print(tokens.word_ids())

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]


In [39]:
def align_labels_with_tokens(labels, word_ids):
    aligned_labels = []
    previous_word_id = None
    for word_id in word_ids:
        if word_id is None:
            # special token → -100
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            # 新 word 的第一个 subword → 用原始标签
            aligned_labels.append(labels[word_id])
        else:
            # 同一个 word 的后续 subword → -100
            aligned_labels.append(-100)
        previous_word_id = word_id
    return aligned_labels

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
    )

    all_labels = []
    for i, labels in enumerate(examples['tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned = align_labels_with_tokens(labels, word_ids)
        all_labels.append(aligned)

    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=['tokens', 'tags'],
)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [41]:
print(tokenized_datasets['train'].column_names)
print(tokenized_datasets['train'][0].keys())
print(tokenized_datasets['train'][0])


['input_ids', 'token_type_ids', 'attention_mask', 'labels']
dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 1, 0, 2, 0, 0, 0, 2, 0, -100, 0, -100]}


In [58]:
import numpy as np
!pip install seqeval --quiet

from seqeval.metrics import classification_report, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred # logits dimension (batch, seq_len, num_classes)
    predictions = np.argmax(logits, axis=-1) # (batch, seq_len)

    true_labels = []
    true_predictions = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_pred = []
        true_label = []

        for pred, label in zip(pred_seq, label_seq):
            if label != -100:
                true_pred.append(label_names[pred])
                true_label.append(label_names[label])

        true_predictions.append(true_pred)
        true_labels.append(true_label)

    print(classification_report(true_labels, true_predictions))


    results = {
        'f1': f1_score(true_labels, true_predictions),
    }
    return results

In [43]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


In [47]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

training_arguments = TrainingArguments(
    output_dir="./ner-bert-conll2003",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    logging_steps=100,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Epoch,Training Loss,Validation Loss,F1
1,0.052824,0.045111,0.919189
2,0.027463,0.037955,0.944687
3,0.014582,0.035909,0.947677


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2634, training_loss=0.09535655324985878, metrics={'train_runtime': 276.5717, 'train_samples_per_second': 152.304, 'train_steps_per_second': 9.524, 'total_flos': 1103242267752120.0, 'train_loss': 0.09535655324985878, 'epoch': 3.0})

In [54]:
predictions = trainer.predict(tokenized_datasets["test"])
print(predictions.metrics)


{'test_loss': 0.10655824095010757, 'test_f1': 0.9098735066760365, 'test_runtime': 3.2678, 'test_samples_per_second': 1056.684, 'test_steps_per_second': 33.05}


In [50]:
print(len(tokenized_datasets['test']))

3453


In [55]:
from transformers import pipeline

ner_pipe = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # 合并同一实体的 subword
)

results = ner_pipe("Elon Musk founded SpaceX in Hawthorne, California.")
for entity in results:
    print(f"{entity['word']:20s} {entity['entity_group']:8s} {entity['score']:.3f}")

# Elon Musk            PER      0.998
# SpaceX               ORG      0.995
# Hawthorne            LOC      0.990
# California           LOC      0.997

El                   PER      0.945
##on Musk            PER      0.916
SpaceX               ORG      0.973
Hawthorne            LOC      0.987
California           LOC      0.980


In [62]:
trainer = Trainer(
    model=model,
    args=training_arguments,
    data_collator=data_collator,
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,  # 这次用新的
)
predictions = trainer.predict(tokenized_datasets["test"])


              precision    recall  f1-score   support

         LOC       0.93      0.93      0.93      1668
        MISC       0.78      0.83      0.80       702
         ORG       0.87      0.91      0.89      1661
         PER       0.97      0.95      0.96      1617

   micro avg       0.90      0.92      0.91      5648
   macro avg       0.89      0.90      0.90      5648
weighted avg       0.90      0.92      0.91      5648

